# Traditional ML Live Trade Notebook

This notebook builds a live trade plan for the current `raw_pct6` long-only strategy and can map the resulting target underlyings into Interactive Brokers OTM call orders.

What it does:
- rebuild the `10B+` US equity feature panel
- train the traditional ML stack on all labeled history currently available
- score the latest available market date
- apply the stateful `top_k` strategy rules to current holdings
- output a target portfolio and a `buy / sell / hold` blotter for the next session
- optionally connect to Interactive Brokers, inspect existing call positions, and build an OTM call order blotter

Assumptions:
- broker order submission is disabled by default and must be explicitly enabled in config
- the strategy is long-only
- entries require all configured components to clear the threshold
- exits happen when the classifier flips short or the latest row is invalid
- holdings persist until an exit condition fires; rank alone does not force an exit
- the default option mapping uses the OTM synthetic-research assumptions: calls only, about `90` DTE, strike near `1.05x` spot


In [1]:
import os
from types import SimpleNamespace

import pandas as pd
from IPython.display import display

import django
from django.apps import apps

os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "settings")
if not apps.ready:
    django.setup()

from fmp.workflows import run_scoring_data_refresh_from_fmp
from features.macro import MacroFeatureConfig
from pipeline.api import build_fundamental_dataframe, build_macro_dataframe, build_label_dataframe
from data.preparation import MLDatasetConfig, prepare_ml_dataset
from ml.raw_stack import save_raw_stack_artifacts, train_rf_models, train_ae
from backtest.raw_stack import ProbabilityColumnConfig, enrich_scored_panel
from backtest.latest import run_latest_prediction_custom, make_autoencoder_familiarity_predictor
from app.live_trade_leaderboard import build_latest_scoring_panel
from pipeline.universe_selection import DEFAULT_US_EXCHANGES, resolve_symbol_universe
from pipeline.symbol_filters import select_top_symbols_by_latest_market_cap
from trading.live_trade import (
    build_live_trade_plan,
    build_technical_dataframe_from_django,
    component_cols_for_score,
    expected_latest_price_date_from_market_clock,
    normalize_holdings,
    resolve_fmp_api_key,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)



In [2]:
APP_CFG = {
    "dates": {
        "data_start": "1990-01-01",
        "data_end": pd.Timestamp.today().strftime("%Y-%m-%d"),
    },
    "universe": {
        "country": "US",
        "exchanges": list(DEFAULT_US_EXCHANGES),
        "min_market_cap": 10_000_000_000.0,
        "exclude_pooled_vehicles": True,
        "size": None,
        "top_market_cap_n": 500,
    },
    "labels": {
        "k_params": {"YE": [1, 2, 4, 8]},
        "use_sample_weight": True,
        "alpha": 4.0,
        "r_clip": 0.10,
        "horizon_balance": True,
    },
    "costs": {
        "fee_bps": 5.0,
        "slippage_bps": 5.0,
    },
    "runtime": {
        "rf_split_ratio": 1.0,
        "artifact_dir": os.path.abspath("artifacts/raw_stack"),
    },
    "fmp_refresh": {
        "enabled": bool((lambda now_et: now_et.weekday() < 5 and (now_et.hour >= 17))(pd.Timestamp.now(tz="America/New_York"))),
        "refresh_symbol_sections_before_build": True,
        "refresh_macro_before_build": True,
        "mode": "scoring_ready",
        "skip_cached_inactive_symbols": True,
        "skip_recent_price_attempts": True,
        "existing_historical_sections_only": True,
        "max_symbols": None,
        "verbose": True,
    },
    "probability_columns": {
        "buy_col": "clf__prob_1",
        "short_col": None,
        "infer_short_from_buy": True,
    },
    "strategy": {
        "score_col": "buy_score_mean_raw_pct6",
        "component_threshold": 0.50,
        "top_k": 20,
        "price_col": "close",
        "instrument": "otm_call",
    },
    "options": {
        "tenor_days": 90,
        "min_days_to_expiry": 45,
        "max_days_to_expiry": 90,
        "strike_multiplier": 1.05,
        "right": "C",
        "exchange": "SMART",
        "currency": "USD",
        "contracts_per_position": 1,
        "size_positions_by_target_weight": True,
        "max_contracts_per_position": None,
        "underlying_price_col": "close",
        "include_market_data": True,
        "market_data_type": 3,
        "order_type": "CLOSEPX",
        "close_px_base_order_type": "MKT",
        "close_px_max_pct_vol": 0.1,
        "close_px_risk_aversion": "Neutral",
        "close_px_start_time": "12:00:00 US/Eastern",
        "close_px_force_completion": True,
        "limit_price_slippage_pct": 0.05,
    },
    "broker": {
        "enabled": True,
        "host": "127.0.0.1",
        "port": 4002,
        "client_id": 17,
        "account": "",
        "load_existing_positions": True,
        "cancel_open_orders_on_connect": True,
        "cancel_wait_seconds": 1.0,
        "submit_orders": True,
    },
}

# Optional manual underlying override. If broker position loading is enabled, IBKR positions take precedence.
CURRENT_HOLDINGS = []

display(pd.DataFrame([
    {
        "data_start": APP_CFG["dates"]["data_start"],
        "data_end": APP_CFG["dates"]["data_end"],
        "min_market_cap": APP_CFG["universe"]["min_market_cap"],
        "score_col": APP_CFG["strategy"]["score_col"],
        "component_threshold": APP_CFG["strategy"]["component_threshold"],
        "top_k": APP_CFG["strategy"]["top_k"],
        "instrument": APP_CFG["strategy"]["instrument"],
        "fmp_refresh_enabled": APP_CFG["fmp_refresh"]["enabled"],
        "refresh_symbol_sections_before_build": APP_CFG["fmp_refresh"]["refresh_symbol_sections_before_build"],
        "refresh_macro_before_build": APP_CFG["fmp_refresh"]["refresh_macro_before_build"],
        "fmp_api_key_configured": bool(resolve_fmp_api_key(required=False)),
        "tenor_days": APP_CFG["options"]["tenor_days"],
        "size_positions_by_target_weight": APP_CFG["options"]["size_positions_by_target_weight"],
        "strike_multiplier": APP_CFG["options"]["strike_multiplier"],
        "broker_enabled": APP_CFG["broker"]["enabled"],
        "submit_orders": APP_CFG["broker"]["submit_orders"],
        "current_holdings": ", ".join(CURRENT_HOLDINGS) or "<none>",
    }
]))




,data_start,data_end,min_market_cap,score_col,component_threshold,top_k,instrument,fmp_refresh_enabled,refresh_symbol_sections_before_build,refresh_macro_before_build,fmp_api_key_configured,tenor_days,size_positions_by_target_weight,strike_multiplier,broker_enabled,submit_orders,current_holdings
0,1990-01-01,2026-05-14,1.000000e+10,buy_score_mean_raw_pct6,0.5,20,otm_call,False,True,True,True,90,True,1.05,True,True,<none>


In [ ]:
START_DATE = APP_CFG["dates"]["data_start"]
END_DATE = APP_CFG["dates"]["data_end"]
EXPECTED_PRICE_DATE = pd.Timestamp(expected_latest_price_date_from_market_clock()).normalize()
if pd.Timestamp(END_DATE).normalize() > EXPECTED_PRICE_DATE:
    END_DATE = EXPECTED_PRICE_DATE.strftime("%Y-%m-%d")
    APP_CFG["dates"]["data_end"] = END_DATE

ctx = SimpleNamespace(api_key=resolve_fmp_api_key(required=False))
universe = tuple(
    resolve_symbol_universe(
        min_market_cap=float(APP_CFG["universe"]["min_market_cap"]),
        country=str(APP_CFG["universe"]["country"]),
        exchanges=list(APP_CFG["universe"]["exchanges"]),
        exclude_pooled_vehicles=bool(APP_CFG["universe"]["exclude_pooled_vehicles"]),
        limit=APP_CFG["universe"]["size"],
    )
)
if not universe:
    raise RuntimeError("No symbols resolved for the configured universe.")

APP_CFG.setdefault("fmp_refresh", {})["enabled"] = bool((lambda now_et: now_et.weekday() < 5 and (now_et.hour >= 17))(pd.Timestamp.now(tz="America/New_York")))
fmp_refresh_cfg = dict(APP_CFG.get("fmp_refresh", {}))
fmp_refresh_enabled = bool(fmp_refresh_cfg.get("enabled", False))
ctx = SimpleNamespace(api_key=resolve_fmp_api_key(required=False))
macro_feature_config = MacroFeatureConfig()
fmp_price_refresh_summary = pd.DataFrame()
fmp_symbol_refresh_summary = pd.DataFrame()
fmp_macro_refresh_summary = pd.DataFrame()
fmp_refresh_ran = False

if fmp_refresh_enabled and bool(resolve_fmp_api_key(required=False)):
    fmp_refresh_ran = True
    fmp_refresh_results = run_scoring_data_refresh_from_fmp(
        symbols=universe,
        target_start_date=START_DATE,
        target_end_date=END_DATE,
        refresh_mode=str(fmp_refresh_cfg.get("mode") or "scoring_ready"),
        refresh_symbol_sections_before_build=bool(fmp_refresh_cfg.get("refresh_symbol_sections_before_build", False)),
        refresh_macro_before_build=bool(fmp_refresh_cfg.get("refresh_macro_before_build", False)),
        skip_cached_inactive_symbols=bool(fmp_refresh_cfg.get("skip_cached_inactive_symbols", True)),
        max_symbols=fmp_refresh_cfg.get("max_symbols"),
        existing_historical_sections_only=bool(fmp_refresh_cfg.get("existing_historical_sections_only", True)),
        macro_config=macro_feature_config,
        verbose=bool(fmp_refresh_cfg.get("verbose", True)),
        progress_logger=print,
    )
    fmp_price_refresh_summary = fmp_refresh_results.get("price_refresh_results", pd.DataFrame())
    fmp_symbol_refresh_summary = fmp_refresh_results.get("fundamental_refresh_results", pd.DataFrame())
    if fmp_symbol_refresh_summary.empty:
        fmp_symbol_refresh_summary = fmp_refresh_results.get("symbol_refresh_results", pd.DataFrame())
    fmp_macro_refresh_summary = fmp_refresh_results.get("macro_refresh_results", pd.DataFrame())
    price_skipped_inactive = int((fmp_price_refresh_summary["status"] == "skipped_inactive").sum()) if not fmp_price_refresh_summary.empty and "status" in fmp_price_refresh_summary.columns else 0
    price_skipped_recent = int((fmp_price_refresh_summary["status"] == "skipped_recent_attempt").sum()) if not fmp_price_refresh_summary.empty and "status" in fmp_price_refresh_summary.columns else 0
    symbol_skipped_fresh = int((fmp_symbol_refresh_summary["status"] == "skipped_fresh").sum()) if not fmp_symbol_refresh_summary.empty and "status" in fmp_symbol_refresh_summary.columns else 0
    symbol_empty_fetch = int(((fmp_symbol_refresh_summary["status"] == "ok") & pd.to_numeric(fmp_symbol_refresh_summary.get("sections_fetched"), errors="coerce").fillna(0).eq(0)).sum()) if not fmp_symbol_refresh_summary.empty and "status" in fmp_symbol_refresh_summary.columns else 0
    combined_skipped_total = int(price_skipped_inactive + price_skipped_recent + symbol_skipped_fresh)
    if not fmp_price_refresh_summary.empty or not fmp_symbol_refresh_summary.empty:
        display(pd.DataFrame([
            {
                "symbols_attempted": int(len(fmp_price_refresh_summary) + len(fmp_symbol_refresh_summary)),
                "symbols_skipped_total": combined_skipped_total,
                "symbols_skipped_inactive": price_skipped_inactive,
                "symbols_skipped_recent": price_skipped_recent,
                "symbols_skipped_fresh": symbol_skipped_fresh,
                "symbols_with_empty_fetch": symbol_empty_fetch,
                "price_records_fetched": int(pd.to_numeric(fmp_price_refresh_summary.get("records_fetched"), errors="coerce").fillna(0).sum()),
                "symbol_sections_fetched": int(pd.to_numeric(fmp_symbol_refresh_summary.get("sections_fetched"), errors="coerce").fillna(0).sum()),
                "symbol_sections_skipped": int(pd.to_numeric(fmp_symbol_refresh_summary.get("sections_skipped"), errors="coerce").fillna(0).sum()),
                "retry_attempts": int(pd.to_numeric(fmp_price_refresh_summary.get("retries_used"), errors="coerce").fillna(0).sum()) + int(pd.to_numeric(fmp_symbol_refresh_summary.get("retry_attempts"), errors="coerce").fillna(0).sum()),
            }
        ]))
    if not fmp_price_refresh_summary.empty:
        display(pd.DataFrame([
            {
                "symbols_attempted": int(len(fmp_price_refresh_summary)),
                "symbols_ok": int((fmp_price_refresh_summary["status"] == "ok").sum()),
                "symbols_skipped_total": int(price_skipped_inactive + price_skipped_recent),
                "symbols_skipped_inactive": price_skipped_inactive,
                "symbols_skipped_recent": price_skipped_recent,
                "symbols_skipped_recent": price_skipped_recent,
                "symbols_failed": int((fmp_price_refresh_summary["status"] == "error").sum()) if "status" in fmp_price_refresh_summary.columns else 0,
                "records_fetched": int(pd.to_numeric(fmp_price_refresh_summary.get("records_fetched"), errors="coerce").fillna(0).sum()),
                "retry_attempts": int(pd.to_numeric(fmp_price_refresh_summary.get("retries_used"), errors="coerce").fillna(0).sum()),
            }
        ]))
    if not fmp_symbol_refresh_summary.empty:
        display(pd.DataFrame([
            {
                "symbols_attempted": int(len(fmp_symbol_refresh_summary)),
                "symbols_ok": int((fmp_symbol_refresh_summary["status"] == "ok").sum()),
                "symbols_skipped_total": symbol_skipped_fresh,
                "symbols_skipped_fresh": symbol_skipped_fresh,
                "symbols_with_empty_fetch": symbol_empty_fetch,
                "symbols_with_errors": int((fmp_symbol_refresh_summary["status"] == "completed_with_errors").sum()),
                "symbols_failed": int((fmp_symbol_refresh_summary["status"] == "error").sum()),
                "sections_total": int(pd.to_numeric(fmp_symbol_refresh_summary.get("sections_total"), errors="coerce").fillna(0).sum()),
                "sections_fetched": int(pd.to_numeric(fmp_symbol_refresh_summary.get("sections_fetched"), errors="coerce").fillna(0).sum()),
                "sections_skipped": int(pd.to_numeric(fmp_symbol_refresh_summary.get("sections_skipped"), errors="coerce").fillna(0).sum()),
                "partial_sections": int(pd.to_numeric(fmp_symbol_refresh_summary.get("partial_sections"), errors="coerce").fillna(0).sum()),
                "retry_attempts": int(pd.to_numeric(fmp_symbol_refresh_summary.get("retry_attempts"), errors="coerce").fillna(0).sum()),
                "avg_symbol_duration_s": round(float(pd.to_numeric(fmp_symbol_refresh_summary.get("duration_s"), errors="coerce").dropna().mean() or 0.0), 3),
            }
        ]))
    if not fmp_macro_refresh_summary.empty:
        display(fmp_macro_refresh_summary)

technical_df, technical_cols = build_technical_dataframe_from_django(
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
)
if technical_df.empty:
    raise RuntimeError("No technical feature rows were built.")

technical_dates = pd.DatetimeIndex(technical_df.index.get_level_values("date")).normalize()
technical_symbols = pd.Index(technical_df.index.get_level_values("symbol"))
technical_latest_date = pd.Timestamp(technical_dates.max()).normalize()
technical_coverage_by_date = pd.Series(technical_symbols, index=technical_dates).groupby(level=0).nunique().tail(10)
expected_price_rows = int(technical_coverage_by_date.get(EXPECTED_PRICE_DATE, 0))
print(f"Expected latest completed market date: {EXPECTED_PRICE_DATE.date().isoformat()}")
print(f"Latest local technical price date: {technical_latest_date.date().isoformat()}")
display(
    technical_coverage_by_date.rename("symbols_with_price_rows")
    .reset_index()
    .rename(columns={"index": "date"})
)
if technical_latest_date < EXPECTED_PRICE_DATE or expected_price_rows < len(universe):
    latest_by_symbol = (
        pd.DataFrame({"symbol": technical_symbols, "date": technical_dates})
        .sort_values(["symbol", "date"])
        .groupby("symbol", as_index=False, sort=False)
        .tail(1)
    )
    stale_latest = latest_by_symbol.loc[latest_by_symbol["date"].lt(EXPECTED_PRICE_DATE)].copy()
    present_symbols = set(latest_by_symbol["symbol"].astype(str).str.upper())
    missing_symbols = [symbol for symbol in universe if str(symbol).strip().upper() not in present_symbols]
    stale_sample = ", ".join(
        f"{row.symbol}:{pd.Timestamp(row.date).date().isoformat()}"
        for row in stale_latest.head(20).itertuples(index=False)
    )
    missing_sample = ", ".join(missing_symbols[:20])
    print(
        f"FMP/local price coverage is incomplete for {EXPECTED_PRICE_DATE.date().isoformat()}; "
        "symbols without target-date data will use their own latest available Scored Date. "
        f"{expected_price_rows:,}/{len(universe):,} symbols have target-date technical rows. "
        f"First stale symbols: {stale_sample or '<none>'}. "
        f"First missing symbols: {missing_sample or '<none>'}."
    )

fund_df, fund_cols = build_fundamental_dataframe(
    ctx=ctx,
    symbols=universe,
    start_date=START_DATE,
    end_date=END_DATE,
    target_index=technical_df.index,
    daily_prices=technical_df,
    verbose=False,
)
macro_df, macro_cols = build_macro_dataframe(
    ctx=ctx,
    start_date=START_DATE,
    end_date=END_DATE,
    config=macro_feature_config,
    target_index=technical_df.index,
    verbose=False,
)

final_df = pd.concat([technical_df, fund_df, macro_df], axis=1).sort_index()
top_market_cap_n = APP_CFG["universe"].get("top_market_cap_n")
if top_market_cap_n not in (None, ""):
    market_cap_selection = select_top_symbols_by_latest_market_cap(
        final_df,
        end_date=END_DATE,
        top_n=int(top_market_cap_n),
        symbols=universe,
    )
    selected_universe = tuple(
        str(symbol).strip().upper()
        for symbol in market_cap_selection.get("selected_symbols", [])
        if str(symbol).strip()
    )
    if selected_universe:
        universe = selected_universe
        technical_df = technical_df.loc[technical_df.index.get_level_values("symbol").isin(universe)].copy()
        fund_df = fund_df.loc[fund_df.index.get_level_values("symbol").isin(universe)].copy()
        macro_df = macro_df.loc[macro_df.index.get_level_values("symbol").isin(universe)].copy()
        final_df = final_df.loc[final_df.index.get_level_values("symbol").isin(universe)].copy()

latest_feature_date = pd.Timestamp(final_df.index.get_level_values("date").max())

EXECUTION_PARAMS = {
    "price_col": "close",
    "fee_bps": float(APP_CFG["costs"]["fee_bps"]),
    "slippage_bps": float(APP_CFG["costs"]["slippage_bps"]),
}
WEIGHTING_PARAMS = {
    "use_sample_weight": bool(APP_CFG["labels"]["use_sample_weight"]),
    "alpha": float(APP_CFG["labels"]["alpha"]),
    "r_clip": float(APP_CFG["labels"]["r_clip"]),
    "horizon_balance": bool(APP_CFG["labels"]["horizon_balance"]),
}

symbols_in_panel = set(technical_df.index.get_level_values("symbol"))
daily_map_all = {
    s: technical_df.xs(s, level="symbol").copy()
    for s in universe
    if s in symbols_in_panel
}
label_df_all = build_label_dataframe(
    daily_by_symbol=daily_map_all,
    k_params=dict(APP_CFG["labels"]["k_params"]),
    execution_params=EXECUTION_PARAMS,
    weighting=WEIGHTING_PARAMS,
    add_rank_labels=True,
    verbose=False,
)
if label_df_all.empty:
    raise RuntimeError("No labels were generated from the available history.")

train_df, raw_feature_list, _ = prepare_ml_dataset(
    features_df=final_df,
    labels_df=label_df_all,
    target_cols=["target", "trade_return", "trade_duration_days"],
    weight_col="sample_weight",
    config=MLDatasetConfig(drop_nan_features=False),
    verbose=True,
)
if train_df.empty:
    raise RuntimeError("The joined training dataset is empty.")

trade_return_values = pd.to_numeric(train_df["trade_return"], errors="coerce")
train_df["trade_return_pct_target"] = trade_return_values.rank(pct=True, method="average")

rf_bundle = train_rf_models(
    train_df,
    raw_feature_list,
    split_ratio=float(APP_CFG["runtime"]["rf_split_ratio"]),
    classifier_target_col="target",
    ranking_target_col="rank_y",
    classifier_market_position_col=None,
    train_trade_return_model=True,
    trade_return_target_col="trade_return_pct_target",
    train_duration_model=False,
)
clf_raw = rf_bundle.clf
reg_raw = rf_bundle.trade_return_reg if rf_bundle.trade_return_reg is not None else rf_bundle.ranking_reg
ae_raw, ae_numeric_cols = train_ae(train_df, raw_feature_list)

saved_artifact_dir = save_raw_stack_artifacts(
    clf_raw=clf_raw,
    reg_trade_return_raw=reg_raw,
    reg_duration_raw=rf_bundle.duration_reg,
    ae_raw=ae_raw,
    raw_feature_list=raw_feature_list,
    ae_raw_numeric_cols=ae_numeric_cols,
    artifact_dir=str(APP_CFG["runtime"]["artifact_dir"]),
)

display(pd.DataFrame([
    {
        "universe_size": len(universe),
        "feature_rows": len(final_df),
        "feature_cols": final_df.shape[1],
        "label_rows": len(label_df_all),
        "train_rows": len(train_df),
        "fmp_refresh_enabled": fmp_refresh_enabled,
        "refresh_symbol_sections_before_build": bool(fmp_refresh_cfg.get("refresh_symbol_sections_before_build", False)),
        "refresh_macro_before_build": bool(fmp_refresh_cfg.get("refresh_macro_before_build", False)),
        "fmp_api_key_configured": bool(ctx.api_key),
        "latest_feature_date": latest_feature_date.date().isoformat(),
        "artifact_dir": str(saved_artifact_dir),
        "saved_duration_regressor": bool(rf_bundle.duration_reg is not None),
    }
]))



Expected latest completed market date: 2026-05-13
Latest local technical price date: 2026-05-07


,date,symbols_with_price_rows
0,2026-04-24,689
1,2026-04-27,689
2,2026-04-28,689
3,2026-04-29,689
4,2026-04-30,689
5,2026-05-01,689
6,2026-05-04,689
7,2026-05-05,688
8,2026-05-06,688
9,2026-05-07,686


FMP/local price coverage is incomplete for 2026-05-13; symbols without target-date data will use their own latest available Scored Date. 0/762 symbols have target-date technical rows. First stale symbols: A:2026-05-07, AA:2026-05-07, ABBV:2026-05-07, ABC:2023-09-25, ABMD:2022-12-23, ABNB:2026-05-07, ABT:2026-05-07, ACM:2026-05-07, ADBE:2026-05-07, ADI:2026-05-07, ADM:2026-05-07, ADP:2026-05-07, ADSK:2026-05-07, AEE:2026-05-07, AEIS:2026-05-07, AEP:2026-05-07, AES:2026-05-07, AESC:2024-02-14, AFG:2026-05-07, AFL:2026-05-07. First missing symbols: RTPY.


In [ ]:
prob_cfg = ProbabilityColumnConfig(**APP_CFG["probability_columns"])
ae_predict = make_autoencoder_familiarity_predictor(ae_numeric_cols)


from trading.ibkr import (
    build_ib_otm_option_trade_plan,
    ensure_ib_connection,
    fetch_ib_open_orders,
    fetch_ib_portfolio_snapshot,
    fetch_ib_option_positions,
    liquidate_all_positions as run_ibkr_liquidation,
    submit_ib_orders,
)


def liquidate_all_positions(*, ib=None, mods=None, broker_cfg=None, option_cfg=None, display_results=True, disconnect_when_done=None):
    effective_broker_cfg = dict(APP_CFG.get("broker", {}))
    if broker_cfg:
        effective_broker_cfg.update(broker_cfg)
    effective_option_cfg = dict(APP_CFG.get("options", {}))
    if option_cfg:
        effective_option_cfg.update(option_cfg)

    result = run_ibkr_liquidation(
        broker_cfg=effective_broker_cfg,
        option_cfg=effective_option_cfg,
        ib=ib if ib is not None else globals().get("ib"),
        mods=mods if mods is not None else globals().get("ib_mods"),
        disconnect_when_done=disconnect_when_done,
    )
    if display_results:
        cancelled_orders = result.get("cancelled_orders", pd.DataFrame())
        if not cancelled_orders.empty:
            print("Cancelled open IBKR orders after connect")
            display(cancelled_orders)
        print("IBKR full liquidation summary")
        display(result["summary"])
        print("IBKR positions to liquidate")
        display(result["positions"].drop(columns=["contract"], errors="ignore"))
        print("IBKR liquidation order blotter")
        display(result["orders"].drop(columns=["contract"], errors="ignore"))
        if not result["submission_results"].empty:
            print("IBKR liquidation submission results")
            display(result["submission_results"])
    return result


def print_current_ib_positions(*, ib=None, broker_cfg=None, display_results=True):
    effective_broker_cfg = dict(APP_CFG.get("broker", {}))
    if broker_cfg:
        effective_broker_cfg.update(broker_cfg)

    ib_instance, _, owns_connection, cancelled_orders = ensure_ib_connection(
        broker_cfg=effective_broker_cfg,
        ib=ib if ib is not None else globals().get("ib"),
        mods=globals().get("ib_mods"),
    )
    try:
        portfolio_df = fetch_ib_portfolio_snapshot(
            ib_instance,
            account=str(effective_broker_cfg.get("account", "")).strip() or None,
        )
        if display_results:
            if not cancelled_orders.empty:
                print("Cancelled open IBKR orders after connect")
                display(cancelled_orders)
            print("Current IBKR positions and PnL")
            display(portfolio_df.drop(columns=["contract"], errors="ignore"))
        return portfolio_df
    finally:
        if owns_connection:
            try:
                ib_instance.disconnect()
            except Exception:
                pass


def print_current_ib_positions_and_orders(*, ib=None, broker_cfg=None, display_results=True):
    effective_broker_cfg = dict(APP_CFG.get("broker", {}))
    if broker_cfg:
        effective_broker_cfg.update(broker_cfg)

    ib_instance, _, owns_connection, cancelled_orders = ensure_ib_connection(
        broker_cfg=effective_broker_cfg,
        ib=ib if ib is not None else globals().get("ib"),
        mods=globals().get("ib_mods"),
    )
    try:
        account = str(effective_broker_cfg.get("account", "")).strip() or None
        portfolio_df = fetch_ib_portfolio_snapshot(ib_instance, account=account)
        open_orders_df = fetch_ib_open_orders(ib_instance, account=account)
        if display_results:
            if not cancelled_orders.empty:
                print("Cancelled open IBKR orders after connect")
                display(cancelled_orders)
            print("Current IBKR positions and PnL")
            display(portfolio_df.drop(columns=["contract"], errors="ignore"))
            print("Current IBKR open orders")
            display(open_orders_df.drop(columns=["contract", "order", "trade"], errors="ignore"))
        return {
            "positions": portfolio_df,
            "open_orders": open_orders_df,
            "cancelled_orders": cancelled_orders,
        }
    finally:
        if owns_connection:
            try:
                ib_instance.disconnect()
            except Exception:
                pass


scoring_panel, scoring_panel_stats = build_latest_scoring_panel(
    feature_df=final_df,
    scoring_date=pd.Timestamp(END_DATE),
)
display(pd.DataFrame([{
    "requested_scoring_date": pd.Timestamp(END_DATE).date().isoformat(),
    "symbols_scored": int(scoring_panel_stats["symbol_count"]),
    "exact_date_rows": int(scoring_panel_stats["exact_date_count"]),
    "carry_forward_rows": int(scoring_panel_stats["carry_forward_count"]),
}]))

latest_date, latest_scored = run_latest_prediction_custom(
    train_data=scoring_panel,
    model_specs=[
        {"model": clf_raw, "pred_col": "clf", "include_class_probs": True},
        {"model": reg_raw, "pred_col": "ranking"},
        {"model": ae_raw, "pred_col": "ae_familiarity", "predict_fn": lambda df, m: ae_predict(df, m)},
    ],
    market_position_value=None,
    combine_scores_fn=lambda df: pd.to_numeric(df[prob_cfg.buy_col], errors="coerce").fillna(0.0)
    * pd.to_numeric(df["ranking"], errors="coerce").fillna(0.0)
    * pd.to_numeric(df["ae_familiarity"], errors="coerce").fillna(1.0),
    row_filter_fn=None,
    round_decimals=None,
)
latest_scored = enrich_scored_panel(latest_scored, prob_config=prob_cfg)

score_col = str(APP_CFG["strategy"]["score_col"])
component_cols = component_cols_for_score(score_col)
broker_cfg = dict(APP_CFG.get("broker", {}))
option_cfg = dict(APP_CFG.get("options", {}))
ib = None
ib_mods = None
ib_cancelled_orders = pd.DataFrame()
current_option_positions = pd.DataFrame()
current_underlyings = normalize_holdings(CURRENT_HOLDINGS)

if bool(broker_cfg.get("enabled", False)):
    ib, ib_mods, _, ib_cancelled_orders = ensure_ib_connection(
        broker_cfg=broker_cfg,
        ib=globals().get("ib"),
        mods=globals().get("ib_mods"),
    )
    if not ib_cancelled_orders.empty:
        print("Cancelled open IBKR orders after connect")
        display(ib_cancelled_orders)
    if bool(broker_cfg.get("load_existing_positions", True)):
        current_option_positions = fetch_ib_option_positions(
            ib,
            account=str(broker_cfg.get("account", "")).strip() or None,
            right=str(option_cfg.get("right", "C")),
        )
        broker_underlyings = normalize_holdings(current_option_positions["symbol"].tolist()) if not current_option_positions.empty else []
        if broker_underlyings:
            current_underlyings = broker_underlyings

live_plan = build_live_trade_plan(
    latest_scored_df=latest_scored,
    current_holdings=current_underlyings,
    top_k=int(APP_CFG["strategy"]["top_k"]),
    score_col=score_col,
    component_cols=component_cols,
    component_threshold=float(APP_CFG["strategy"]["component_threshold"]),
    price_col=str(APP_CFG["strategy"]["price_col"]),
)

summary_df = live_plan["summary"].copy()
summary_df.insert(0, "as_of_date", latest_date.date().isoformat())
summary_df.insert(1, "instrument", str(APP_CFG["strategy"].get("instrument", "equity")))
summary_df.insert(2, "current_underlyings", ", ".join(current_underlyings) or "<none>")

print(f"Live trade plan as of {latest_date.date().isoformat()} (for the next session)")
display(summary_df)

print("Target portfolio")
display(live_plan["target_portfolio"])

print("Trade blotter")
display(live_plan["actions"])

print("Watchlist: eligible longs not currently held")
display(live_plan["watchlist"])

print("Top latest scores")
display(live_plan["latest_scored"].head(25)[[
    score_col,
    "score_rank",
    "close",
    "prob_buy",
    "prob_short",
    "pred_rf_reg",
    "ae_familiarity",
    "prob_buy_pct",
    "pred_rf_reg_pct",
    "ae_familiarity_pct",
]])

ib_option_plan = None
ib_order_results = pd.DataFrame()
if str(APP_CFG["strategy"].get("instrument", "equity")) == "otm_call":
    if ib is None:
        raise RuntimeError("OTM option trading requires an active IBKR connection. Set APP_CFG['broker']['enabled'] = True and rerun.")
    ib_option_plan = build_ib_otm_option_trade_plan(
        ib,
        ib_mods,
        live_plan=live_plan,
        current_option_positions=current_option_positions,
        option_cfg=option_cfg,
        broker_cfg=broker_cfg,
        as_of_date=latest_date,
        instrument=str(APP_CFG["strategy"].get("instrument", "equity")),
    )
    print("IBKR option trade summary")
    display(ib_option_plan["summary"])
    print("Current IBKR option positions")
    display(ib_option_plan["current_option_positions"].drop(columns=["contract"], errors="ignore"))
    print("Desired OTM call contracts")
    display(ib_option_plan["desired_contracts"].drop(columns=["contract"], errors="ignore"))
    skipped_symbols_df = ib_option_plan.get("skipped_symbols", pd.DataFrame())
    if not skipped_symbols_df.empty:
        print("Skipped symbols with no eligible option contract")
        display(skipped_symbols_df)
    print("IBKR order blotter")
    display(ib_option_plan["orders"].drop(columns=["contract"], errors="ignore"))
    ib_order_results = submit_ib_orders(ib, ib_mods, ib_option_plan["orders"], broker_cfg, option_cfg)
    if not ib_order_results.empty:
        print("IBKR order submission results")
        display(ib_order_results)


In [ ]:
#liquidate_all_positions()

In [ ]:
summary_df.head()